# Chapter 8 · Delegating with isolation

Three tasks that touch the same files, for three agents at once. Four ways of splitting them are compared: each in its own copy with the last one winning, all in the same copy, in waves, and each in its own copy with an integrator that merges three ways and only lets in what leaves the tests green. The agent is the one of chapter 7, with frame, plan and reviewer.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. Sections 7 and 8 can be done without a GPU.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 8: delegating with isolation."""
import ast
import importlib
import io
import json
import re
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
GREEDY = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
SAMPLING = {"do_sample": True, "temperature": 0.7, "top_p": 0.8,
            "top_k": 20}           # the values Qwen recommends for this model
decoding = dict(GREEDY)            # the one generate() uses; run_all() changes it
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with the current decoding: greedy or with
    sampling. Returns the text and the input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, **decoding, max_new_tokens=MAX_TOKENS_STEP,
                                prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. The rules of chapter 2 and the contract of chapter 4

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name


def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer, require_test=True,
                allows_docs=None):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer. With "require_test" False,
    R7 does not ask for a new check; "allows_docs", if given, decides R3 instead of
    the request."""
    if allows_docs is None:
        allows_docs = bool(ASKS_DOCS.search(task["request"]))
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not allows_docs:
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test or not require_test,
             "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not self.allows_docs():
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists:
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def allows_docs(self):
        """True if the request asks to change the terms and conditions (R3)."""
        return bool(ASKS_DOCS.search(self.task["request"]))

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result


def contract_text(contract):
    """A contract (like data/tasks/Txx/contract.json) written for the agent: goal,
    paths it may change, forbidden paths and numbered acceptance criteria (C1,
    C2…)."""
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    return ("Task contract\n"
            f"Goal: {contract['goal']}\n"
            f"You may change: {', '.join(contract['allowed_paths'])}\n"
            f"Don't touch: {', '.join(contract['forbidden_paths'])}\n"
            f"Acceptance criteria:\n{criteria}\n"
            "The contract overrides AGENTS.md: if it forbids tests/, don't add tests.")


def forbids_tests(contract):
    """True if the contract forbids changing tests/ (for example, a refactoring
    that must pass the tests without touching them)."""
    return rt.path_matches("tests/x.py", contract["forbidden_paths"])


def check_contract(contract, changes, answer):
    """What breaks the contract at the end: files outside the allowed paths or in
    the forbidden ones, and criteria the answer does not explain how it meets."""
    broken = {}
    paths = rt.check_scope(changes["changed"], contract)
    outside = sorted(set(paths["out_of_scope"]) | set(paths["forbidden"]))
    if outside:
        broken["Scope"] = (f"you changed {', '.join(outside)}, which the contract "
                           "does not allow")
    numbers = range(1, len(contract["acceptance_criteria"]) + 1)
    unexplained = [f"C{i}" for i in numbers
                   if not re.search(rf"\bC{i}\b", answer or "")]
    if unexplained:
        broken["Criteria"] = ("your answer does not say how it meets "
                              + ", ".join(unexplained)
                              + " (cite each criterion by its number)")
    return broken


class ToolsWithContract(ToolsWithRules):
    """The tools with the rules of chapter 2 and, on top of them, the contract:
    nothing can be edited or created outside the allowed paths."""

    def __init__(self, tools, task, contract):
        super().__init__(tools, task)
        self.contract = contract

    def allows_docs(self):
        """The contract overrides R3: docs/ can be changed if the contract allows it."""
        return rt.path_matches("docs/terms_and_conditions.md",
                               self.contract["allowed_paths"])

    def broken_rule(self, name, path):
        """The rule or the part of the contract the edit would break, or None."""
        contract = self.contract
        if rt.path_matches(path, contract["forbidden_paths"]):
            return f"Contract: {path} is among the paths you must not touch."
        if not rt.path_matches(path, contract["allowed_paths"]):
            return (f"Contract: {path} is not among the paths you may change ("
                    + ", ".join(contract["allowed_paths"]) + ").")
        return super().broken_rule(name, path)

## 4. The feedback and the verification gate of chapter 5

In [ ]:
MAX_FAILURES = 3   # failures summarized for the agent


def summarize_failures(output, folder):
    """The failures of a unittest run, one per test: which test fails, the lines of
    the repository the error goes through (without Python's own) and the last line,
    the one with the error. Paths are left relative to the copy of the repository."""
    failures = []
    prefix = str(folder) + "\\" if "\\" in str(folder) else str(folder) + "/"
    for block in re.split(r"^={50,}$", output, flags=re.M)[1:]:
        header = re.search(r"^(FAIL|ERROR): (\S+)", block, re.M)
        if not header:
            continue
        lines = block.split("-" * 70)[1] if "-" * 70 in block else block
        at = []
        for m in re.finditer(r'File "([^"]+)", line (\d+), in (\S+)\n\s+(.+)', lines):
            if m.group(1).startswith(prefix):
                path = m.group(1)[len(prefix):].replace("\\", "/")
                at.append(f"{path}:{m.group(2)} ({m.group(3)}): {m.group(4).strip()}")
        error = [l for l in lines.strip().splitlines() if l.strip()][-1:]
        failures.append({"test": f"{header.group(1)} {header.group(2)}",
                         "where": at[-3:], "error": error[0].strip() if error else ""})
    return failures[:MAX_FAILURES]


class ToolsWithFeedback(ToolsWithContract):
    """The tools with the contract and, besides, feedback: after each successful
    edit the tests are run and the result of the edit carries a summary of the
    failures. run_tests returns the same summary."""

    def tests(self):
        """The repository's tests, summarized for the agent."""
        result = self.tools.run_tests()
        summary = {"all_pass": result["all_pass"],
                   "result": result["result"]}
        if not result["all_pass"]:
            summary["failures"] = summarize_failures(result["output"],
                                                     self.tools.workspace.folder)
        return summary

    def call(self, name, arguments):
        """Like ToolsWithContract.call, with the tests after each edit."""
        if name == "run_tests":
            result = self.tests()
            self.log.append({"tool": name, "arguments": {}, "ok": True,
                             "seconds": 0, "result": result})
            return result
        result = super().call(name, arguments)
        if name in ("edit", "create_file") and "error" not in result:
            result = dict(result, tests_after_edit=self.tests())
        return result


def code_without_docstrings(text):
    """The syntax tree of a program, as text, without its docstrings. Comments never
    reach the tree, so two versions that differ only in comments or docstrings give
    the same result. None if it cannot be parsed."""
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return None
    for node in ast.walk(tree):
        body = getattr(node, "body", None)
        if (isinstance(body, list) and body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant)
                and isinstance(body[0].value.value, str)):
            node.body = body[1:] or [ast.Pass()]
    return ast.dump(tree)


def changes_code(workspace, path):
    """True if the file has changed in something other than comments and docstrings."""
    original, new = workspace.source / path, workspace.path(path)
    if not path.endswith(".py") or not original.exists() or not new.exists():
        return True
    before = code_without_docstrings(original.read_text(encoding="utf-8"))
    after = code_without_docstrings(new.read_text(encoding="utf-8"))
    return before is None or after is None or before != after


def verification_gate(workspace, contract):
    """What is missing to accept the task, looking at the repository and not at what
    the agent says: code changes that are not only comments, a new test that fails
    with the original code (if the contract allows touching tests/) and all the
    tests green. Returns a list of sentences; empty if the work can be accepted."""
    changes = rt.differences(workspace.source, workspace.folder)
    lines = diff_lines(changes["diff"])
    missing = []
    code = [r for r in changes["changed"]
            if not r.startswith(("tests/", "docs/", "data/"))]
    if not any(changes_code(workspace, r) for r in code):
        missing.append("the program code has not changed (comments don't count)")
    new_tests = [r for r, l in lines.items() if r.startswith("tests/")
                 and any(re.search(r"def test_|assert", x) for x in l["+"])]
    if not forbids_tests(contract):
        if not new_tests:
            missing.append("there is no new test in tests/")
        else:
            with rt.Workspace() as original:    # the new tests, with the old code
                for path in new_tests:
                    destination = original.path(path)
                    destination.write_text(workspace.path(path).read_text("utf-8"),
                                           encoding="utf-8")
                before = rt.Tools(original).run_tests()
            if before["all_pass"]:
                missing.append("your new tests also pass with the original code, so "
                               "they don't check the change the request asks for")
    now = rt.Tools(workspace).run_tests()
    if not now["all_pass"]:
        failures = summarize_failures(now["output"], workspace.folder)
        missing.append("the tests don't pass: "
                       + "; ".join(f"{f['test']} → {f['error']}" for f in failures))
    return missing

## 5. The reviewer of chapter 6 and the frame and plan of chapter 7

In [ ]:
MAX_REVIEWER_STEPS = 10  # model calls in each review


MAX_REVIEWS = 2          # times the reviewer can send the work back


ESCALATED = ("NOT DONE: the reviewer has rejected the change twice; it is waiting "
             "for a person to review it.")


READ_ONLY = ("list_files", "read_file", "search", "run_tests")


REVIEWER_SYSTEM = (
    "You are the change reviewer of the North Cycles repository, a bike store in "
    "Portland, Oregon. Another agent has made a change to solve a request and says "
    "it is done. Your job is to check whether the change really solves the "
    "request. You can list, read and search files and run the tests, but you "
    "cannot change anything. Use one tool at a time. When you are done, answer "
    "without calling any tool, with only a JSON.")


ASK_FREE = """Review this change.

Request:
{request}

Change (diff):
{diff}

What the author says:
{answer}

At the end, answer with this JSON:
{{"approved": true or false, "comments": ["…"]}}"""


ASK_RUBRIC = """Review this change criterion by criterion.

Request:
{request}

Acceptance criteria:
{criteria}

Change (diff):
{diff}

What the author says:
{answer}

For each criterion, look in the code (not in what the author says) to see whether
it is met, and give the evidence: file and line, or the test that checks it. Also
check whether the change touches anything the request does not ask for. At the end,
answer with this JSON:
{{"criteria": [{{"id": "C1", "met": true or false, "evidence": "…"}}],
  "beyond_request": ["…"], "approved": true or false}}
Approve only if every criterion is met."""


class ReadOnlyTools:
    """The reviewer's tools: the usual ones, but only those that do not change
    anything. Any other one is rejected."""

    def __init__(self, tools):
        self.tools = tools
        self.log = []

    def call(self, name, arguments):
        """Like Tools.call, only for the reading tools."""
        if name not in READ_ONLY:
            result = {"error": f'The reviewer cannot use "{name}": only '
                               + ", ".join(READ_ONLY) + "."}
        else:
            result = self.tools.call(name, arguments)
        self.log.append({"tool": name, "ok": "error" not in result})
        return result


def read_verdict(answer):
    """The last JSON object of the reviewer's answer, or None if there is none
    with the key "approved"."""
    for piece in reversed(re.findall(r"\{.*\}", answer, re.S)):
        try:
            verdict = json.loads(piece, strict=False)
        except ValueError:
            continue
        if isinstance(verdict, dict) and "approved" in verdict:
            return verdict
    return None


def review(tok, model, task, contract, workspace, author_answer, rubric, verbose=False):
    """One review: a separate agent, with its own conversation and read-only tools
    on the copy of the repository, that returns its verdict. Without a readable
    verdict, the change is approved (the reviewer does not block for failing)."""
    changes = rt.differences(workspace.source, workspace.folder)
    criteria = "\n".join(f"C{i}. {c}" for i, c in
                         enumerate(contract["acceptance_criteria"], 1))
    request_template = ASK_RUBRIC if rubric else ASK_FREE
    messages = [{"role": "system", "content": REVIEWER_SYSTEM},
                {"role": "user", "content": request_template.format(
                    request=task["request"], criteria=criteria,
                    diff=changes["diff"][:3000], answer=author_answer[:800])}]
    tools = ReadOnlyTools(rt.Tools(workspace))
    verdict, steps = None, 0
    for steps in range(1, MAX_REVIEWER_STEPS + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, _, _ = generate(tok, model, text)
        calls = read_calls(answer)
        if not calls:
            verdict = read_verdict(answer)
            break
        messages.append({"role": "assistant", "content": answer.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            result = (tools.call(call["name"], call["arguments"])
                      if call["name"] else {"error": "Invalid call."})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    if verbose:
        print(f"    REVIEWER ({steps} steps): "
              f"{json.dumps(verdict, ensure_ascii=False)[:300]}")
    return {"approved": True if verdict is None else bool(verdict.get("approved")),
            "verdict": verdict, "steps": steps,
            "calls": len(tools.log)}


def comments_for_author(verdict):
    """What the reviewer objects to, in sentences for the author."""
    if not verdict:
        return "the reviewer does not approve the change"
    sentences = [f"{c.get('id', '?')} is not met: {c.get('evidence', '')}"
                 for c in verdict.get("criteria", []) if not c.get("met", True)]
    sentences += [f"beyond the request: {x}" for x in verdict.get("beyond_request", [])]
    sentences += [str(x) for x in verdict.get("comments", [])]
    return "; ".join(sentences) or "the reviewer does not approve the change"

In [ ]:
MAX_FRAME_STEPS = 10   # model calls to explore and write the frame


ASK_FRAME = """Before changing anything, understand the task. Explore the repository
with the reading tools and write the task frame in JSON, with nothing after it:
{{"goal": "one sentence",
  "facts": [{{"fact": "what you have checked", "evidence": "file:line"}}],
  "paths": ["files that will have to change"],
  "unknowns": ["what you don't know and how you would find out"]}}
Each fact must cite a file and a line that prove it.

Request:
{request}

{contract}"""


ASK_PLAN = """Now write the plan to solve it, in JSON, with nothing after it:
{{"steps": [{{"step": "what to do", "file": "where", "check": "which test shows it"}}]}}
Each step must say how you will check that it is done right."""


def read_json(answer, key):
    """The last JSON object of an answer that has the given key, or None."""
    for piece in reversed(re.findall(r"\{.*\}", answer, re.S)):
        try:
            data = json.loads(piece, strict=False)
        except ValueError:
            continue
        if isinstance(data, dict) and key in data:
            return data
    return None


def check_evidence(frame, workspace):
    """The facts of the frame whose evidence does not exist: a file that is not
    there or a line it does not have."""
    bad = []
    for fact in frame.get("facts", []):
        m = re.match(r"\s*([\w./-]+\.\w+)(?::(\d+))?", str(fact.get("evidence", "")))
        file = workspace.path(m.group(1)) if m else None
        if not m or not file.is_file():
            bad.append(f'"{fact.get("fact", "")}": {fact.get("evidence")} '
                       "does not exist")
        elif m.group(2) and int(m.group(2)) > len(
                file.read_text(encoding="utf-8").splitlines()):
            bad.append(f'"{fact.get("fact", "")}": {m.group(1)} has no line '
                       f"{m.group(2)}")
    return bad


def converse(tok, model, messages, tools, max_steps, key, verbose=False):
    """A conversation with reading tools that ends with a JSON with "key".
    Returns the JSON (or None) and the steps."""
    for steps in range(1, max_steps + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            return None, steps
        answer, _, _ = generate(tok, model, text)
        calls = read_calls(answer)
        if not calls:
            messages.append({"role": "assistant", "content": answer})
            return read_json(answer, key), steps
        messages.append({"role": "assistant", "content": answer.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            result = (tools.call(call["name"], call["arguments"])
                      if call["name"] else {"error": "Invalid call."})
            if verbose:
                print(f"  [frame {steps}] {call['name']}"
                      f"({json.dumps(call['arguments'], ensure_ascii=False)[:80]})")
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    return None, max_steps


def frame_task(tok, model, task, workspace, with_plan, verbose=False):
    """The prior phase: the agent explores without changing anything and writes the
    task frame; the program checks the evidence and, if any does not exist, sends it
    back once. With "with_plan", it also writes a plan with a check per step.
    Returns the text
    that is added to the request and what has happened."""
    contract = task["contract"]
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": ASK_FRAME.format(
                    request=task["request"], contract=contract_text(contract))}]
    reading = ReadOnlyTools(rt.Tools(workspace))
    frame, steps = converse(tok, model, messages, reading, MAX_FRAME_STEPS,
                            "goal", verbose)
    bad = check_evidence(frame, workspace) if frame else []
    if bad:
        messages.append({"role": "user", "content":
                         "This evidence does not exist: " + "; ".join(bad)
                         + ". Fix the frame and write it again in JSON."})
        fixed, more = converse(tok, model, messages, reading, 4, "goal", verbose)
        steps += more
        frame = fixed or frame
        bad = check_evidence(frame, workspace) if frame else bad
    plan = None
    if with_plan and frame:
        messages.append({"role": "user", "content": ASK_PLAN})
        plan, more = converse(tok, model, messages, reading, 2, "steps", verbose)
        steps += more
    parts = []
    if frame:
        parts.append("Task frame (written before starting):\n"
                     + json.dumps(frame, ensure_ascii=False, indent=1)[:2500])
    if plan:
        parts.append("Plan:\n" + json.dumps(plan, ensure_ascii=False, indent=1)[:1500])
    return "\n\n".join(parts), {"frame": frame, "plan": plan, "steps": steps,
                                "bad_evidence": bad}

## 6. An agent that works in turns

`work` is a generator: it hands back control after each call to the model.

In [ ]:
def work(tok, model, task, workspace, frame=False, plan=False, reviewer=None,
         max_steps=MAX_STEPS, verbose=False):
    """The agent of chapter 7 (contract, feedback, verification gate, reviewer and,
    if asked for, frame and plan) on the copy of the repository it is given. It is a
    generator: it hands back control after each call to the model, so that several
    agents can work in turns. When it finishes, it returns what happened."""
    contract = task["contract"]
    request = task["request"] + "\n\n" + contract_text(contract)
    phase = {"frame": None, "plan": None, "steps": 0, "bad_evidence": []}
    if frame:
        prior, phase = frame_task(tok, model, task, workspace, with_plan=plan,
                                  verbose=verbose)
        if prior:
            request += "\n\n" + prior
        yield
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + AGENTS_RULES},
                {"role": "user", "content": request}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    reviews = []
    base = rt.Tools(workspace)
    tools = ToolsWithFeedback(base, task, contract)
    for steps in range(1, max_steps + 1):
        text, trimmed = trim_conversation(tok, messages)
        if trimmed is None:
            break
        answer, n_input, n_output = generate(tok, model, text)
        n_in, n_out = n_in + n_input, n_out + n_output
        yield   # here the other agents can work
        calls = read_calls(answer)
        if not calls:
            messages.append({"role": "assistant", "content": answer})
            if verbose:
                print(f"[{task['id']} {steps}] {answer}")
            if says_done(answer) and rejections < MAX_REJECTIONS:
                changes = rt.differences(workspace.source, workspace.folder)
                broken = check_rules(task, base.log, changes, answer,
                                     require_test=not forbids_tests(contract),
                                     allows_docs=tools.allows_docs())
                broken.update(check_contract(contract, changes, answer))
                broken.pop("R7", None)
                missing = verification_gate(workspace, contract)
                if missing:
                    broken["Verification"] = "; ".join(missing)
                if broken:
                    rejections += 1
                    messages.append({"role": "user", "content":
                                     "It isn't done yet: "
                                     + "; ".join(f"{r}: {t}" for r, t in
                                                 sorted(broken.items()))
                                     + ". Keep working."})
                    continue
            if reviewer and says_done(answer):
                if len(reviews) == MAX_REVIEWS:   # it has already rejected it twice
                    final = ESCALATED
                    break
                this_review = review(tok, model, task, contract, workspace, answer,
                                     rubric=reviewer == "rubric", verbose=verbose)
                reviews.append(this_review)
                if not this_review["approved"]:
                    messages.append({"role": "user", "content":
                                     "The reviewer does not approve the change: "
                                     + comments_for_author(this_review["verdict"])
                                     + ". Fix it and answer DONE again."})
                    continue
            final = answer
            break
        thought = answer.split("<tool_call>")[0].strip()
        messages.append({"role": "assistant", "content": thought,
                         "tool_calls": [{"type": "function", "function": c}
                                        for c in calls if c["name"]]})
        for call in calls:
            if call["name"] is None:
                result = {"error": "The tool call cannot be understood."}
            else:
                result = tools.call(call["name"], call["arguments"])
            if verbose:
                print(f"[{task['id']} {steps}] {call['name']}"
                      f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                      f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
    return {"final": final, "steps": steps, "input_tokens": n_in,
            "output_tokens": n_out, "rejections": rejections, "reviews": reviews,
            "phase": phase, "log": base.log}


def run_to_end(generator):
    """Advances a generator until it finishes and returns its final value."""
    while True:
        try:
            next(generator)
        except StopIteration as end:
            return end.value


def read_changes(workspace):
    """The files that have changed in a copy, with their final text (None if it has
    been deleted)."""
    changes = rt.differences(workspace.source, workspace.folder)
    return {r: None if r in changes["deleted"] else workspace.path(r).read_text(
                encoding="utf-8") for r in changes["changed"]}


def agent(tok, model, task, source=None, verbose=False, **options):
    """An agent in its own copy of the repository: a copy of the original or, with
    "source", of another folder, for example what has already been integrated.
    Returns the row of results, with the files it has changed and how they ended."""
    start = time.time()
    with (rt.Workspace(source) if source else rt.Workspace()) as workspace:
        result = run_to_end(work(tok, model, task, workspace, verbose=verbose,
                                 **options))
        evaluation = rt.evaluate_attempt(workspace, task)
        changes = read_changes(workspace)
    done = result["final"] is not None and says_done(result["final"])
    return {"task": task["id"], "solved": evaluation["solved"],
            "says_done": done,
            "false_done": done and not evaluation["solved"],
            "visible": evaluation["visible"], "hidden": evaluation["hidden"],
            "out_of_scope": evaluation["out_of_scope"],
            "rejections": result["rejections"], "reviews": len(result["reviews"]),
            "steps": result["steps"] + result["phase"]["steps"],
            "input_tokens": result["input_tokens"],
            "seconds": round(time.time() - start, 1),
            "answer": result["final"] or "(no final answer)", "changes": changes}

## 7. Integrate

It can be tried without the model, with the reference solutions: those of T15 and T25 clash in the terms and conditions; those of T30 and T20 merge without conflicts, but break T20.

In [ ]:
def merge_file(base, ours, theirs):
    """Three-way merge of a file with git merge-file: what there was when the task
    started (base), what has been integrated so far (ours) and what the task has
    done (theirs). Returns the merged text and whether there was a conflict."""
    with tempfile.TemporaryDirectory() as folder:
        paths = []
        for name, text in (("ours", ours), ("base", base), ("theirs", theirs)):
            path = Path(folder) / name
            path.write_bytes(text.encode("utf-8"))   # in bytes: the line endings
            paths.append(str(path))                  # do not change on Windows
        r = subprocess.run(["git", "merge-file", "-p", "-L", "integrated", "-L",
                            "original", "-L", "task", *paths], capture_output=True)
    return r.stdout.decode("utf-8"), r.returncode != 0


def integrate(integrated, base, changes, merge=True):
    """Brings the changes of a task into the integrated copy. Without merging, it
    copies the files as they are and the last one wins. Merging, each file that
    already existed is merged three ways with what has been integrated. Returns the
    files with a conflict."""
    conflicts = []
    for relative, text in changes.items():
        destination, original = integrated.path(relative), Path(base) / relative
        if text is None:
            destination.unlink(missing_ok=True)
            continue
        if merge and original.exists() and destination.exists():
            text, conflict = merge_file(original.read_text(encoding="utf-8"),
                                        destination.read_text(encoding="utf-8"), text)
            if conflict:
                conflicts.append(relative)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(text.encode("utf-8"))
    return conflicts


def snapshot(integrated, paths):
    """The current text of some files of the integrated copy, to be able to go back."""
    return {r: integrated.path(r).read_text(encoding="utf-8")
            if integrated.path(r).exists() else None for r in paths}


def green(workspace):
    """Whether all the visible tests of a copy pass."""
    return rt.Tools(workspace).run_tests()["all_pass"]

In [ ]:
def with_the_solution(task_id):
    """The changes of a task's reference solution, without the model."""
    with rt.Workspace() as workspace:
        rt.apply_solution(rt.Tools(workspace),
                          rt.read_task(task_id)["folder"] / "solution")
        return read_changes(workspace)

for pair in (("T15", "T25"), ("T30", "T20")):
    with rt.Workspace() as integrated:
        conflicts = [c for i in pair
                     for c in integrate(integrated, integrated.source,
                                        with_the_solution(i))]
        hidden = {i: rt.run_hidden(integrated.folder, rt.read_task(i))["pass"]
                  for i in pair}
    print(pair, "conflicts:", conflicts, "· hidden tests:", hidden)

## 8. Four ways to split a batch

By default two tasks of the orders batch are run, each in its own copy, and they are put together in two ways: copying the changes one after another and with the integrator. With `COMPLETE = True`, the whole batch with the four ways. After that, the saved results of the three full runs.

In [ ]:
BATCHES = {"orders": ["T22", "T28", "T29"], "invoices": ["T04", "T26", "T12"],
           "stock": ["T24", "T23", "T05"], "notices": ["T30", "T11", "T20"],
           "shipping": ["T15", "T25", "T17"]}


AGENT = {"frame": True, "plan": True, "reviewer": "rubric"}   # the one of chapter 7


def batch_result(workspace, tasks, steps, critical_path, **others):
    """How the batch ends up: the visible tests of the final copy, the hidden ones of
    each task, the steps of all the agents and those of the longest path."""
    hidden = {t["id"]: rt.run_hidden(workspace.folder, t)["pass"] for t in tasks}
    return {"visible": green(workspace), "hidden": hidden,
            "standing": sum(hidden.values()), "steps": steps,
            "critical_path": critical_path, **others}


def same_copy(tok, model, tasks, verbose=False):
    """The tasks at the same time, in turns, in the same copy of the repository."""
    with rt.Workspace() as workspace:
        agents = {t["id"]: work(tok, model, t, workspace, verbose=verbose, **AGENT)
                  for t in tasks}
        outcomes = {}
        while agents:
            for task_id in list(agents):
                try:
                    next(agents[task_id])
                except StopIteration as end:
                    outcomes[task_id] = end.value
                    del agents[task_id]
        steps = {i: o["steps"] + o["phase"]["steps"] for i, o in outcomes.items()}
        return batch_result(workspace, tasks, sum(steps.values()),
                            max(steps.values()))


def separately(tok, model, tasks, verbose=False):
    """Each task in its own copy of the original, all from the same starting point."""
    return [agent(tok, model, t, verbose=verbose, **AGENT) for t in tasks]


def last_wins(tasks, rows):
    """The changes of each copy, copied one after another onto the original."""
    with rt.Workspace() as integrated:
        for row in rows:
            integrate(integrated, integrated.source, row["changes"], merge=False)
        return batch_result(integrated, tasks, sum(r["steps"] for r in rows),
                            max(r["steps"] for r in rows))


def with_integrator(tok, model, tasks, rows, verbose=False):
    """Each task in its copy, and an integrator that merges their changes one by one.
    Only what leaves the tests green gets in: if a merge has conflicts or breaks a
    test, it undoes it and asks for the task again, now on what has already been
    integrated; if it fails again, it leaves it for a person."""
    steps, critical = sum(r["steps"] for r in rows), max(r["steps"] for r in rows)
    conflicts, redone, left_out = 0, [], []
    with rt.Workspace() as integrated:
        for task, row in zip(tasks, rows):
            before = snapshot(integrated, row["changes"])
            clashes = integrate(integrated, integrated.source, row["changes"])
            conflicts += len(clashes)
            if not clashes and green(integrated):
                continue
            integrate(integrated, integrated.folder, before, merge=False)   # undo
            again = agent(tok, model, task, source=integrated.folder, verbose=verbose,
                          **AGENT)
            steps, critical = steps + again["steps"], critical + again["steps"]
            redone.append(task["id"])
            before = snapshot(integrated, again["changes"])
            integrate(integrated, integrated.folder, again["changes"])
            if not green(integrated):
                integrate(integrated, integrated.folder, before, merge=False)
                left_out.append(task["id"])
        return batch_result(integrated, tasks, steps, critical, conflicts=conflicts,
                            redone=redone, left_out=left_out)


def share_files(a, b):
    """Whether two tasks may touch the same file, according to the paths of their
    contracts."""
    return bool(set(a["contract"]["allowed_paths"])
                & set(b["contract"]["allowed_paths"]))


def in_waves(tok, model, tasks, rows, verbose=False):
    """Exclusive paths: two tasks that share files do not work at the same time.
    They are split into waves; the tasks of a wave each work in their own copy of
    what has already been integrated, and at the end they are all integrated. The
    first wave starts from the original, so its rows are those of separately."""
    waves = []
    for task in tasks:
        free = next((w for w in waves if not any(share_files(task, t) for t in w)),
                    None)
        if free is None:
            waves.append([task])
        else:
            free.append(task)
    done = {r["task"]: r for r in rows}
    steps = critical = 0
    with rt.Workspace() as integrated:
        for number, wave in enumerate(waves):
            if number:
                done.update({t["id"]: agent(tok, model, t, source=integrated.folder,
                                            verbose=verbose, **AGENT) for t in wave})
            base = integrated.source if number == 0 else integrated.folder
            for task in wave:
                integrate(integrated, base, done[task["id"]]["changes"])
            steps += sum(done[t["id"]]["steps"] for t in wave)
            critical += max(done[t["id"]]["steps"] for t in wave)
        return batch_result(integrated, tasks, steps, critical, waves=len(waves))


def run_all(tok, model, seed=None, batches=BATCHES, verbose=False):
    """The batches with the four ways to split them. Without a seed, with greedy
    decoding; with a seed, with sampling, and each part with its own seed."""
    decoding.clear()
    decoding.update(GREEDY if seed is None else SAMPLING)
    results = {}
    for name, task_ids in batches.items():
        tasks = [rt.read_task(i) for i in task_ids]
        number = list(BATCHES).index(name) if name in BATCHES else 0

        def sow(part):
            """Sets the seed of one part of the batch, if there is a seed."""
            if seed is not None:
                torch.manual_seed(1000 * seed + 10 * number + part)

        sow(0)
        rows = separately(tok, model, tasks, verbose)
        sow(1)
        batch = {"separately": [{k: v for k, v in r.items() if k != "changes"}
                                for r in rows],
                 "same copy": same_copy(tok, model, tasks, verbose),
                 "last wins": last_wins(tasks, rows)}
        sow(2)
        batch["with integrator"] = with_integrator(tok, model, tasks, rows, verbose)
        sow(3)
        batch["in waves"] = in_waves(tok, model, tasks, rows, verbose)
        results[name] = batch
        print(f"  {name:12s} " + "  ".join(
            f"{k}: {v['standing']}/3" for k, v in batch.items() if k != "separately")
            + f"  (in its own copy: {sum(r['solved'] for r in rows)}/3)", flush=True)
    return results

In [ ]:
# With COMPLETE = False two tasks are run with two ways (about 35 minutes on
# Colab's T4); with True, the whole batch with the four ways (about three and a
# half hours on the T4).
COMPLETE = False
if COMPLETE:
    results = run_all(tok, model, seed=1,
                      batches={"orders": BATCHES["orders"]})["orders"]
else:
    decoding.clear()
    decoding.update(SAMPLING)
    torch.manual_seed(1)
    tasks = [rt.read_task(i) for i in ("T28", "T29")]
    rows = separately(tok, model, tasks)
    results = {"last wins": last_wins(tasks, rows),
               "with integrator": with_integrator(tok, model, tasks, rows)}
for form, r in results.items():
    if form != "separately":
        print(f"{form:15s} standing {r['standing']}/{len(r['hidden'])}  "
              f"tests green {r['visible']}")

In [ ]:
saved = json.loads((ROOT / "data" / "results" / "chapter_8"
                    / "results_repetitions.json").read_text(encoding="utf-8"))
for key, batch in saved.items():
    for form, r in batch.items():
        if form != "separately":
            print(f"{key:22s} {form:15s} standing {r['standing']}/3")

## Exercises

The results files the exercises cite are in `data/results/chapter_8/`, which `prepare` downloads with the data.

1. What is the difference between a textual conflict and a semantic one? Give an example of each with the North Cycles tasks. *(No GPU.)*

2. With `results_repetitions.json`, find in the orders batch which tasks are solved in their own copy and are not still standing with "the last one wins". Which file erased them? *(No GPU.)*

3. Apply the reference solutions of T15 and T25, which are in `data/tasks/Txx/solution`, on two copies of the repository and merge the terms and conditions with `merge_file`. Resolve the conflict by hand. What would an agent need to know to resolve it by itself? *(No GPU.)*

4. Do the same with the solutions of T30 and T20, which merge without conflicts, and run the hidden tests of both. Write a visible test that would have caught the semantic conflict. *(No GPU.)*

5. Count in the results how many tasks the integrator asked for again, how many came out right the second time and how many were left out. For the ones left out, what would you tell the person who receives them? *(No GPU.)*

6. If each call to the model takes three seconds and each agent has its own GPU, how long does each way of splitting take according to its path? And with a single GPU? Which one would you choose in each case? *(No GPU.)*

7. Combine the waves with the integrator's check: one task after another on what has already been integrated, but only what leaves the tests green gets in. Try it with the orders batch. The results of one run are in `results_exercises.json`. *(GPU, about 50 minutes, or no GPU with the saved results.)*

8. In the same copy, the agents work in strict turns. Describe what would happen if two agents read `orders.py`, thought at the same time and wrote one after the other. Which tool of chapter 2 would partly prevent it? *(No GPU.)*

9. `share_files` decides that two tasks clash if their contracts allow some common file. Propose a finer way, by functions instead of files, and say which tasks of the five batches could then work at the same time. *(No GPU.)*

10. **Project.** In a repository of your own, give three tasks at once to a coding agent, each one in its own `git worktree`, and integrate them into a branch with continuous integration. Write down how many get in the first time, how many clash and how long it takes you to resolve each clash.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [git merge-file](https://git-scm.com/docs/git-merge-file). [Continuous Integration](https://martinfowler.com/articles/continuousIntegration.html) (Martin Fowler). The North Cycles store, its program and its tasks are fictional. Original code and experiments.